In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from  langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, ToolMessage

import os 
load_dotenv()
if os.environ["GROQ_API_KEY"]:
    print("load env variable")
else:
    raise ValueError("not loaded")

load env variable


In [4]:
llm = ChatGroq(
    model_name="qwen/qwen3.6-27b",
    temperature=0.7
)

llm.invoke("I want to know the meaning of water").content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Query:** "I want to know the meaning of water"\n   - **Key Concept:** "water"\n   - **Intent:** The user is asking for the "meaning" of water. This is ambiguous. It could mean:\n     - Scientific/chemical definition\n     - Biological/ecological importance\n     - Cultural/symbolic/spiritual significance\n     - Linguistic/etymological meaning\n     - Philosophical meaning\n   - **Goal:** Provide a comprehensive, well-structured response that covers the multiple dimensions of "meaning" while being clear, accurate, and accessible.\n\n2.  **Identify Key Dimensions to Cover:**\n   - Scientific/Chemical: H₂O, physical properties, states of matter\n   - Biological/Ecological: Essential for life, solvent, habitat, climate regulation\n   - Cultural/Symbolic: Purity, life, transformation, spirituality across cultures/religions\n   - Linguistic/Etymological: Origin of the word\n   - Philosophical/Practical: Resource,

### **Tools** ###

### **DuckDuckGo Search Tool** ###

In [5]:

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
@tool
def duck_research(query: str) -> str:
    """us this tool only for Searches the current or live data on the web using DuckDuckGo and returns the results."""
    api_wrapper = DuckDuckGoSearchAPIWrapper(time="d", max_results=5)
    duck_research = DuckDuckGoSearchResults(api_wrapper=api_wrapper)
    return duck_research.invoke(query)

### **Arxiv Query Tool** ###

In [6]:

from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

@tool 
def arxiv_research(query: str) -> str:
    """only use this tool to Searches the research paper using arxiv and returns the results."""
    api_wrapper = ArxivAPIWrapper()
    arxiv_query = ArxivQueryRun(api_wrapper=api_wrapper)
    return arxiv_query.invoke(query)


### **Wikipedia Search Tool** ###

In [7]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wikipedia_search(query: str) -> str:
    """only use this tool when searching for the latest articles on Wikipedia."""
    api_wrapper = WikipediaAPIWrapper()
    wikipedia_query = WikipediaQueryRun(api_wrapper=api_wrapper)
    return wikipedia_query.invoke(query)

### **Custom Tools**  ###

In [8]:

@tool
def personal_info(name:str):
    """use this tool only when need to get personal information of a person"""
    info = {
        "kartik": "kartik is a software engineer with 5 years of experience in web development.",
        "udit": "udit is a data scientist who specializes in machine learning and AI.",
        "sourabh": "Sourabh is a graphic designer with a passion for creating visually appealing designs.",
    }
    return info.get(name, "Information not available for the given name.")


### **Tool Binding** ###

In [9]:
tools = [duck_research, arxiv_research, wikipedia_search, personal_info]
llm_with_tools = llm.bind_tools(tools)


# **LangGraph Creation**

### **create schema**

In [10]:
from typing import TypedDict, List 

class graph_schema(TypedDict):
    messages: List

### **Create Node Functions**

In [14]:
def llm_node(state: graph_schema) -> graph_schema:
    message = state["message"]

    # Prompt template for the LLM, including system instructions and the human input
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that can use tools to answer questions."),
        ("human",{input})
     ])

     # LLM With Tools 
    chain =  prompt | llm_with_tools   
    response = chain.invoke({"input": message})
      
      # Update the state with the new message
    state["mesages"] = message + [response]
    return state 


### **ToolNode**

In [ ]:
def tool_node(state:graph_schema)->graph_schema:
    messages = state["messages"]
    tool_by_name = {tool.name:tool for tool in tools}
    tool_result =[]
    for tool in messages[-1].tool_calls:
        tool = tool_by_name[tool["name"]]
        observations = tool.invoke(tool["args"])
        tool_result.append(ToolMessage(content=observations,tool_call_id=tool["id"]))
        state["messages"] = messages + tool_result

### **Create condition edge funcation**

In [ ]:
from langgraph.graph import StateGraph,START,END
def if_tool_call(state:graph_schema)->str:
    last_message = state["message"][-1]
    if last_message.tool_calls:
        return "tool_node"
    else:
        return END

### **Create state graph**

In [ ]:

graph = StateGraph(graph_schema)
graph.add_node("LLM_Node",llm_node)
graph.add_node("Tool_Node",tool_node)

# Add edges here 
graph.add_edge(START,"LLM_Node")
graph.add_conditional_edges("LLM_Node",if_tool_call)
graph.add_edge("Tool_Node","LLM_Node")
graph.add_edge("LLM_Node",END)

ReAct_graph = graph.compile()
# You could see the errors with the below command
Image(first_graph.get_graph().draw_mermaid_png())

# You can use the below command to see the graph without errors
# print(first_graph.get_graph().draw_mermaid())